# DART 공시 정리

`dart_collector.ipynb`가 받아 둔 원본(`source/dart/raw/`)을 읽어 **그래프 DB에 넣을 표**로 정리한다. 받을 자료와 이유는 `graphDB.md` 3장을 따른다.

- 이 노트북은 **서버에 요청하지 않는다.** 받아 둔 파일만 읽으므로 몇 번을 다시 실행해도 된다.
- 어느 파일을 읽을지는 `source/dart/selected_reports.csv`(기업·항목마다 고른 최신 보고서)를 따른다.
- 모든 행에 근거 공시의 **접수번호**(`rcept_no`)와 **기준일**(`base_date`, 보고서 기간이 끝나는 날)을 남긴다. 자료마다 기준일이 다를 수 있다(예: 종속회사는 2025-12-31 사업보고서, 타법인 출자는 2026-06-30 반기보고서).
- 이 단계에서는 **거르지 않고 모두 남긴다.** 단순투자 제외(`purpose_group`), 개인 주주 제외(`is_corp`) 같은 규칙은 그래프를 만들 때 이 표의 열로 거른다.
- 칸을 잘못 채워 말이 안 되는 값(지분율이 0~100%를 벗어남)은 빈칸으로 둔다.

**만드는 표** (모두 `source/dart/parsed/` 아래, UTF-8 BOM CSV)

| 절 | 표 | 출처 | 한 줄 |
|---|---|---|---|
| 3 | `companies.csv` | 기업개황 `company` + `targets.csv` | 대상 기업 하나 |
| 4 | `subsidiaries.csv` | 사업보고서 원문 "XII. 상세표 – 연결대상 종속회사 현황(상세)" | 종속회사 하나 |
| 5 | `associates.csv` | 사업보고서 원문 연결재무제표 주석 "관계기업 및 공동기업 투자" | 관계기업·공동기업 하나 |
| 6 | `investments.csv` | 타법인 출자현황 `otrCprInvstmntSttus` | 출자 대상 하나 |
| 7 | `shareholders.csv` | 최대주주 현황 `hyslrSttus` | 주주 하나(주식 종류별) |
| 8 | `decisions.csv` | 양수·양도, 합병·분할 결정 5종 | 결정 공시 하나 |
| 9 | `parse_summary.csv` | 위 표들 | 기업별 행 수 |
| 10 | `affiliates.csv`, `affiliate_groups.csv` | 사업보고서 원문 "IX. 계열회사 등에 관한 사항" | 계열회사 하나 / 보고 회사 하나 |

**공통 열**: `stock_code`, `corp_name`(공시 기업), `name`(상대 회사, 각주 표시를 지운 이름), `name_raw`(원문 그대로), `alias`(이름 안의 영문 약칭, 예: `SEA`), `rcept_no`, `report`, `base_date`

**걸리는 시간**: 약 10분. 대부분 5절(원문 1,063개의 주석 읽기)이다.

**한눈에**

| | |
|---|---|
| 처리 순서에서 | `dart_targets` → `dart_collector` → **`dart_parser`** → `dart_business_parser`, `name_unifier`, `graph_db` |
| 외부 요청·비용 | 없음(받아 둔 원본만 읽는다) |
| 다시 실행 | 언제든 된다 |

## 1. 설정

In [1]:
# ── 설정: 폴더, 대상 기업, 고른 보고서 ──
import json
import re
import warnings
import zipfile
from collections import Counter
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning

warnings.filterwarnings('ignore', category=XMLParsedAsHTMLWarning)
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.width', 200)

# 이 노트북은 graph_db 폴더에서 실행한다.
ROOT = Path.cwd()
assert ROOT.name == 'graph_db', 'graph_db 폴더에서 실행해야 한다.'

RAW = ROOT / 'source' / 'dart' / 'raw'                          # dart_collector가 받은 원본
SELECTED_PATH = ROOT / 'source' / 'dart' / 'selected_reports.csv'
TARGETS_FILE = ROOT / 'source' / 'targets.csv'
OUT = ROOT / 'source' / 'dart' / 'parsed'                       # 이 노트북이 만드는 표
OUT.mkdir(parents=True, exist_ok=True)

targets = pd.read_csv(TARGETS_FILE, dtype={'stock_code': str})
selected = pd.read_csv(SELECTED_PATH, dtype=str).fillna('')
print('대상 기업', len(targets), '개 / 고른 보고서', len(selected), '줄')
selected['item'].value_counts()

대상 기업 1067 개 / 고른 보고서 3191 줄


item
타법인 출자현황    1064
최대주주 현황     1064
사업보고서 원문    1063
Name: count, dtype: int64

## 2. 공통 함수

**이름 정리** (`clean_name`, `split_alias`)

| 원문 | `name` | `alias` |
|---|---|---|
| `삼성바이오로직스㈜(*2)(*3)` | `삼성바이오로직스㈜` | |
| `Samsung Electronics America, Inc. (SEA)` | `Samsung Electronics America, Inc.` | `SEA` |
| `에프알엘코리아(비상장)` | `에프알엘코리아` | |
| `(주)유한코스메틱\n(보통주)` | `(주)유한코스메틱` | |

- 각주 표시(`(*1)`, `주1)`, `*2`), 상장 여부·주식 종류 표시, 줄바꿈과 겹친 공백을 지운다.
- 회사 이름을 하나로 묶는 일(㈜와 (주), 한글명과 영문명)은 하지 않는다. 결정할 것 #16(회사 이름 통일)에서 정한다.

**나라 정하기** (`country_of`): 주소나 소재지 칸에서 나라 이름을 찾는다. 한국 주소(시·도 이름)는 `대한민국`, 미국 주 이름·약어는 `미국`으로 본다. 찾지 못하면 빈칸이다.

In [2]:
# ── 공통 함수: 저장 파일 읽기, 이름 정리, 숫자·지분율, 기준일 ──
def read_saved(path):
    """dart_collector가 저장한 JSON(받은 시각, 요청 값, 응답을 감싼 형식)을 읽는다."""
    return json.loads(Path(path).read_text(encoding='utf-8'))


def clean(text):
    """줄바꿈과 겹친 공백을 한 칸으로"""
    return re.sub(r'\s+', ' ', str(text or '')).strip()


FOOTNOTE_RE = re.compile(r'\s*(\(\s*\*[\d,\s*\-~]*\)|\(\*\)|\*\d+(,\d+)*|\(?주\s*\d+\))\s*')
LABEL_RE = re.compile(r'\((비?상장|보통주|우선주|상환전환우선주|전환우선주|종류주|구,?\s*[^)]*)\)')
ALIAS_RE = re.compile(r'\s*\(([A-Z][A-Z0-9&\-]{1,11})\)\s*$')


def clean_name(name):
    """회사 이름에서 각주 표시, 상장·주식 종류 표시를 지운다."""
    s = FOOTNOTE_RE.sub(' ', clean(name))
    s = LABEL_RE.sub(' ', s)
    return clean(s).strip(' ,')


def split_alias(name):
    """'Hyundai Motor America (HMA)' → ('Hyundai Motor America', 'HMA')"""
    m = ALIAS_RE.search(name)
    return (name[:m.start()].strip(' ,'), m.group(1)) if m else (name, '')


def to_num(text):
    """'1,234', '(1,234)', '-' 같은 표 숫자를 float로. 숫자가 아니면 None"""
    s = clean(text).replace(',', '')
    if s in ('', '-', '－'):
        return None
    negative = s.startswith('(') and s.endswith(')')
    try:
        v = float(s.strip('()'))
    except ValueError:
        return None
    return -v if negative else v


def valid_ratio(text):
    """지분율(%) 칸 값. 0~100을 벗어나면 칸을 잘못 채운 것이라 None (예: 주식 수 6,440,308을 지분율 칸에 적음)"""
    v = to_num(text)
    return v if v is not None and 0 <= v <= 100 else None


def period_end(report_nm):
    """'사업보고서 (2025.12)' → '2025-12-31' (보고서 기간이 끝나는 날 = 기준일)"""
    m = re.search(r'\((\d{4})\.(\d{2})\)', report_nm)
    if not m:
        return ''
    return str(pd.Period(f'{m.group(1)}-{m.group(2)}', 'M').end_time.date())

In [3]:
# ── 나라 찾기: 주소·소재지 → 나라 ──
# 나라 이름 사전: 정규식 → 나라. 앞에 있을수록 먼저 본다.
COUNTRIES = [
    (r'대한민국|한\s*국|국\s*내|korea|서울|부산|대구|인천|광주광역|대전|울산|세종|경기|강원|충청|충북|충남|전라|전북|전남|경상|경북|경남|제주', '대한민국'),
    (r'미\s*국|u\.?s\.?a\b|united states|\bu\.s\.|america\b|wilmington|silicon valley', '미국'),
    (r'중\s*국|china|\bprc\b|beijing|shanghai|shenzhen|guangzhou|jiangsu|tianjin|북경|상해|천진|소주|강소', '중국'),
    (r'홍\s*콩|hong\s*kong', '홍콩'), (r'대\s*만|taiwan', '대만'), (r'일\s*본|japan|tokyo|osaka', '일본'),
    (r'베트남|vietnam|viet nam|hanoi|ho chi minh', '베트남'), (r'인도네시아|indonesia|jakarta', '인도네시아'),
    (r'싱가포르|싱가폴|singapore', '싱가포르'), (r'말레이시아|malaysia', '말레이시아'), (r'태\s*국|thailand|bangkok', '태국'),
    (r'필리핀|philippines', '필리핀'), (r'인\s*도(?!네)|india\b|mumbai|new delhi|chennai|bangalore', '인도'),
    (r'캄보디아|cambodia', '캄보디아'), (r'미얀마|myanmar', '미얀마'), (r'몽골|mongolia', '몽골'),
    (r'방글라데시|bangladesh', '방글라데시'), (r'파키스탄|pakistan', '파키스탄'), (r'스리랑카|sri lanka', '스리랑카'),
    (r'우즈베키스탄|uzbekistan', '우즈베키스탄'), (r'카자흐스탄|kazakhstan', '카자흐스탄'),
    (r'호\s*주|australia|\b(NSW|VIC|QLD)\s+\d{4}\b|sydney|melbourne', '호주'), (r'뉴질랜드|new zealand', '뉴질랜드'),
    (r'캐나다|canada', '캐나다'), (r'멕시코|mexico', '멕시코'), (r'브라질|brazil|brasil|s[ãa]o paulo|manaus', '브라질'),
    (r'칠레|chile', '칠레'), (r'페루|peru\b', '페루'), (r'아르헨티나|argentina', '아르헨티나'), (r'콜롬비아|colombia', '콜롬비아'),
    (r'파나마|panama', '파나마'),
    (r'영\s*국|united kingdom|\bu\.?k\.?\b|england|london|scotland', '영국'), (r'독\s*일|germany|deutschland|berlin|m[üu]nchen|munich|frankfurt|hamburg|düsseldorf|stuttgart', '독일'),
    (r'프랑스|france|paris', '프랑스'), (r'네덜란드|netherlands|amsterdam', '네덜란드'), (r'벨기에|belgium', '벨기에'),
    (r'룩셈부르크|luxembourg', '룩셈부르크'), (r'스위스|switzerland', '스위스'), (r'오스트리아|austria', '오스트리아'),
    (r'이탈리아|italy', '이탈리아'), (r'스페인|spain', '스페인'), (r'포르투갈|portugal', '포르투갈'),
    (r'아일랜드|ireland', '아일랜드'), (r'스웨덴|sweden', '스웨덴'), (r'노르웨이|norway', '노르웨이'),
    (r'덴마크|denmark', '덴마크'), (r'핀란드|finland', '핀란드'), (r'폴란드|poland', '폴란드'),
    (r'체코|czech', '체코'), (r'슬로바키아|slovak', '슬로바키아'), (r'헝가리|hungary', '헝가리'),
    (r'루마니아|romania', '루마니아'), (r'튀르키예|터키|turkey|t[üu]rkiye', '튀르키예'), (r'러시아|russia|moscow|모스크바', '러시아'),
    (r'우크라이나|ukraine', '우크라이나'),
    (r'아랍에미리트|\bu\.?a\.?e\b|united arab emirates|dubai|두바이|abu dhabi|아부다비', '아랍에미리트'), (r'사우디|saudi|soudi|riyadh|jeddah', '사우디아라비아'),
    (r'카타르|qatar', '카타르'), (r'쿠웨이트|kuwait', '쿠웨이트'), (r'이스라엘|israel', '이스라엘'), (r'이집트|egypt', '이집트'),
    (r'남아프리카|south africa', '남아프리카공화국'), (r'나이지리아|nigeria', '나이지리아'),
    (r'케이[맨먼만]|cayman', '케이맨제도'), (r'버진아일랜드|virgin islands|\bbvi\b', '영국령 버진아일랜드'),
    (r'버뮤다|bermuda', '버뮤다'), (r'마[샬셜]|marshall islands', '마셜제도'), (r'라이베리아|liberia', '라이베리아'),
    (r'라오스|laos|vientiane', '라오스'), (r'오만|\boman\b', '오만'), (r'요르단|jordan', '요르단'), (r'바레인|bahrain', '바레인'),
    (r'이라크|iraq', '이라크'), (r'이란|\biran\b', '이란'), (r'모잠비크|mozambique', '모잠비크'), (r'알제리|algeria', '알제리'),
    (r'가나|ghana', '가나'), (r'케냐|kenya', '케냐'), (r'모로코|morocco', '모로코'), (r'콩고|congo', '콩고'), (r'수단|sudan', '수단'),
    (r'불가리아|bulgaria', '불가리아'), (r'그리스|greece', '그리스'), (r'세르비아|serbia', '세르비아'), (r'크로아티아|croatia', '크로아티아'),
    (r'슬로베니아|slovenia', '슬로베니아'), (r'에스토니아|estonia', '에스토니아'), (r'리투아니아|lithuania', '리투아니아'),
    (r'라트비아|latvia', '라트비아'), (r'몰타|malta', '몰타'), (r'키프로스|cyprus', '키프로스'),
    (r'네팔|nepal', '네팔'), (r'동티모르|timor', '동티모르'), (r'피지|fiji', '피지'), (r'파푸아뉴기니|papua', '파푸아뉴기니'),
    (r'키리바시|kiribati', '키리바시'), (r'투발루|tuvalu', '투발루'), (r'나우루|nauru', '나우루'),
    (r'자메이카|jamaica', '자메이카'), (r'과테말라|guatemala', '과테말라'), (r'코스타리카|costa rica', '코스타리카'),
    (r'에콰도르|ecuador', '에콰도르'), (r'우루과이|uruguay', '우루과이'), (r'파라과이|paraguay', '파라과이'),
    (r'도미니카|dominican', '도미니카공화국'), (r'온두라스|honduras', '온두라스'), (r'엘살바도르|el salvador', '엘살바도르'),
    (r'니카라과|nicaragua', '니카라과'), (r'바하마|bahamas', '바하마'), (r'바베이도스|barbados', '바베이도스'),
    (r'저지|jersey(?!\s*city)', '저지'), (r'건지|guernsey', '건지'), (r'모리셔스|mauritius', '모리셔스'), (r'세이셸|seychelles', '세이셸'),
    (r'튀니지|tunisia', '튀니지'), (r'에티오피아|ethiopia', '에티오피아'), (r'탄자니아|tanzania', '탄자니아'),
    (r'아제르바이잔|azerbaijan', '아제르바이잔'), (r'조지아공화국|tbilisi', '조지아'), (r'키르기스|kyrgyz', '키르기스스탄'),
    (r'브루나이|brunei', '브루나이'), (r'마카오|macau|macao', '마카오'),
]
COUNTRY_RES = [(re.compile(p, re.I), c) for p, c in COUNTRIES]
US_STATE_RE = re.compile(
    r'(?i:\b(Alabama|Alaska|Arizona|Arkansas|California|Colorado|Connecticut|Delaware|Florida|Georgia|Hawaii|Idaho|Illinois|'
    r'Indiana|Iowa|Kansas|Kentucky|Louisiana|Maine|Maryland|Massachusetts|Michigan|Minnesota|Mississippi|Missouri|Montana|'
    r'Nebraska|Nevada|New Hampshire|New Jersey|New Mexico|New York|North Carolina|North Dakota|Ohio|Oklahoma|Oregon|'
    r'Pennsylvania|Rhode Island|South Carolina|South Dakota|Tennessee|Texas|Utah|Vermont|Virginia|Washington|'
    r'West Virginia|Wisconsin|Wyoming)\b)|\b(AL|AK|AZ|AR|CA|CO|CT|DE|FL|GA|HI|ID|IL|IN|IA|KS|KY|LA|ME|MD|MA|MI|MN|MS|MO|'
    r'MT|NE|NV|NH|NJ|NM|NY|NC|ND|OH|OK|OR|PA|RI|SC|SD|TN|TX|UT|VT|VA|WA|WV|WI|WY)\s*\d{5}\b')


def country_of(text):
    """주소·소재지 문자열에서 나라를 찾는다. 주소는 끝에 나라가 오는 경우가 많아 뒤에서 가장 가까운 것을 고른다."""
    s = clean(text)
    if not s or s in ('-', '〃'):
        return ''
    if re.fullmatch(r'[가-힣 ]+', s):        # '베 트 남'처럼 한 글자씩 띄어 쓴 소재지
        s = s.replace(' ', '')
    hits = []
    for pattern, country in COUNTRY_RES:
        for m in pattern.finditer(s):
            hits.append((m.end(), country))
    if US_STATE_RE.search(s):
        hits.append((US_STATE_RE.search(s).end(), '미국'))
    if not hits:
        return ''
    # 한국 주소는 앞에 시·도가 오므로, 한국어로 시작하는 주소는 첫 번째 것을 쓴다
    if re.match(r'[가-힣]', s):
        return min(hits)[1]
    return max(hits)[1]


print(country_of('700 Sylvan Ave, Englewood Cliffs, New Jersey, USA, 07632'),
      country_of('경기도 수원시 영통구 삼성로 129'), country_of('Shanghai, China'), country_of('중 국'), country_of('국 내'))

미국 대한민국 중국 중국 대한민국


## 3. 기업 (`companies.csv`)

대상 기업 하나가 한 줄이다. 기업개황의 영문 정식명칭은 미국 뉴스 검색어를 만들 때 쓴다. 시장·WI26 업종은 `targets.csv`에서 가져온다.

In [4]:
# ── 3. 기업(companies.csv): 기업개황 + 대상 목록 ──
rows = []
for t in targets.itertuples():
    path = RAW / 'company' / f'{t.stock_code}.json'
    if not path.exists():
        continue
    c = read_saved(path)['response']
    rows.append({
        'stock_code': t.stock_code, 'corp_code': c['corp_code'], 'corp_name': c['corp_name'], 'stock_name': c['stock_name'],
        'corp_name_eng': clean(c['corp_name_eng']), 'market': t.market, 'group': t.group, 'wi26_sector': t.wi26_sector,
        'corp_cls': c['corp_cls'], 'induty_code': c['induty_code'], 'est_dt': c['est_dt'], 'acc_mt': c['acc_mt'],
        'hm_url': c['hm_url'], 'adres': clean(c['adres']),
    })
companies = pd.DataFrame(rows)
companies.to_csv(OUT / 'companies.csv', index=False, encoding='utf-8-sig')
CORP_NAME = dict(zip(companies.stock_code, companies.stock_name))   # 종목코드 → 짧은 이름 (아래 표에서 쓴다)
print('기업', len(companies), '개 / 영문명 없는 기업', (companies.corp_name_eng == '').sum(), '개')
companies.head()

기업 1067 개 / 영문명 없는 기업 0 개


,stock_code,corp_code,corp_name,stock_name,corp_name_eng,market,group,wi26_sector,corp_cls,induty_code,est_dt,acc_mt,hm_url,adres
0,005930,00126380,삼성전자(주),삼성전자,"SAMSUNG ELECTRONICS CO,.LTD",KOSPI,코스피200,반도체,Y,264,19690113,12,www.samsung.com/sec,경기도 수원시 영통구 삼성로 129 (매탄동)
1,000660,00164779,에스케이하이닉스(주),SK하이닉스,SK hynix Inc.,KOSPI,코스피200,반도체,Y,2612,19491015,12,www.skhynix.com,경기도 이천시 부발읍 경충대로 2091
2,402340,01596425,에스케이스퀘어 주식회사,SK스퀘어,"SK Square Co., Ltd.",KOSPI,코스피200,반도체,Y,64992,20211102,12,www.sksquare.com,서울특별시 중구 을지로 65 SK T타워
3,373220,01515323,(주)엘지에너지솔루션,LG에너지솔루션,"LG ENERGY SOLUTION, LTD.",KOSPI,코스피200,IT가전,Y,28202,20201201,12,www.lgensol.com,"서울특별시 영등포구 여의대로 108, 타워 1"
4,009150,00126371,삼성전기(주),삼성전기,"SAMSUNG ELECTRO-MECHANICS CO.,LTD",KOSPI,코스피200,IT하드웨어,Y,2622,19730808,12,www.sem.samsung.co.kr,경기도 수원시 영통구 매영로 150


## 4. 종속회사 (`subsidiaries.csv`)

사업보고서 원문의 "XII. 상세표 – 1. 연결대상 종속회사 현황(상세)"에서 뽑는다. 이 표는 공시 서식이 정해져 있어 칸마다 태그(`ACODE`)가 붙어 있다. 그래서 회사마다 표 모양이 달라도 같은 방법으로 읽는다.

| 태그 | 열 | 뜻 |
|---|---|---|
| `CRP_NM` | `name_raw` | 상호 |
| `EST_DT` | `est_date` | 설립일 |
| `ADR` | `address` | 주소. 여기서 `country`를 정한다 |
| `M_IND` | `business` | 주요사업 |
| `KEY_IND` | `total_assets` | 최근 사업연도말 자산총액. 표의 단위(백만원·천원 등)를 원으로 바꾼다 |
| `RUL_REL` | `control_basis` | 지배관계 근거 |
| `M_SUB_COM` | `is_major` | 주요종속회사 여부 |

- 종속회사가 없는 기업은 상호 칸이 `-` 한 줄이라 뺀다.
- 주소 칸이 "상동"이면 위 줄과 같은 나라로 본다. 주소를 "주2)"처럼 표 아래 각주로 미룬 기업은 나라를 알 수 없어 `country`가 빈칸이다.
- 원문 zip은 `selected_reports.csv`의 "사업보고서 원문" 줄이 가리키는 파일(사업보고서 본문이 있는 가장 최근 보고서)을 쓴다.

In [5]:
# ── 4. 종속회사(subsidiaries.csv): 사업보고서 원문 XII. 상세표의 SUB_CMPN 표 ──
docs = selected[selected['item'] == '사업보고서 원문'].copy()
docs['base_date'] = docs['report'].map(period_end)
# 원문 zip 이름의 접수번호 (최신 정정본에 본문이 없어 이전 보고서를 쓴 기업은 latest_rcept_no와 다르다)
docs['rcept_no'] = docs['file'].str.extract(r'_(\d{14})\.zip$', expand=False)


def main_xml(zip_path):
    """원문 zip에서 사업보고서 본문 XML(파일 이름이 접수번호 14자리)을 읽는다. 첨부서류(감사보고서)는 '_'가 붙은 다른 파일이다."""
    with zipfile.ZipFile(zip_path) as z:
        name = next(n for n in z.namelist() if re.fullmatch(r'\d{14}\.xml', n))
        return z.read(name).decode('utf-8', 'replace')


UNIT_WON = {'원': 1, '천원': 1e3, '백만원': 1e6, '억원': 1e8, '십억원': 1e9}


def subsidiary_rows(xml):
    """SUB_CMPN 표의 줄을 {태그: 값}으로 돌려준다. 금액 단위도 함께 돌려준다."""
    group = re.search(r'<TABLE-GROUP ACLASS="SUB_CMPN"[^>]*>(.*?)</TABLE-GROUP>', xml, re.S)
    if not group:
        return [], None
    unit = re.search(r'단위\s*[:：]\s*([^)<]+)', group.group(1))
    unit = UNIT_WON.get(unit.group(1).replace(' ', '')) if unit else None
    out = []
    for tr in re.findall(r'<TR[^>]*>(.*?)</TR>', group.group(1), re.S):
        cells = {k: clean(re.sub(r'<[^>]+>', ' ', v)) for k, v in re.findall(r'<TE[^>]*ACODE="([A-Z_]+)"[^>]*>(.*?)</TE>', tr, re.S)}
        if cells:
            out.append(cells)
    return out, unit


SAME_AS_ABOVE = ('상동', '"', '〃', '″', '동일', '同上', '좌동')

rows, unit_missing = [], []
for d in docs.itertuples():
    xml = main_xml(RAW / d.file)
    found, unit = subsidiary_rows(xml)
    if unit is None:
        unit_missing.append(d.corp_name)
    prev_country = ''
    for c in found:
        if c.get('CRP_NM', '-') in ('', '-', '해당사항 없음', '해당사항없음'):
            continue
        name, alias = split_alias(clean_name(c['CRP_NM']))
        assets = to_num(c.get('KEY_IND'))
        # 주소 칸이 '상동'이면 위 줄과 같은 나라. '주2)'처럼 각주로 미룬 주소는 나라를 알 수 없어 빈칸으로 둔다
        country = prev_country if c.get('ADR', '').strip() in SAME_AS_ABOVE else country_of(c.get('ADR'))
        prev_country = country
        rows.append({
            'stock_code': d.stock_code, 'corp_name': CORP_NAME.get(d.stock_code, d.corp_name),
            'name': name, 'alias': alias, 'name_raw': c['CRP_NM'], 'country': country,
            'address': c.get('ADR', ''), 'business': c.get('M_IND', ''), 'est_date': c.get('EST_DT', ''),
            'total_assets': assets * unit if assets is not None and unit else None,
            'control_basis': c.get('RUL_REL', ''), 'is_major': c.get('M_SUB_COM', '').strip().upper() in ('O', 'Y', '○', '예'),
            'rcept_no': d.rcept_no,
            'report': d.report, 'base_date': d.base_date,
        })

subsidiaries = pd.DataFrame(rows)
subsidiaries.to_csv(OUT / 'subsidiaries.csv', index=False, encoding='utf-8-sig')
print('종속회사', len(subsidiaries), '줄 /', subsidiaries.stock_code.nunique(), '개 기업')
no_country = subsidiaries[subsidiaries.country == '']
footnote_addr = no_country.address.str.fullmatch(r'\(?\*?주?\s*\d*\)?|-|\(\*\d*\)', na=False)
print('나라를 정하지 못한 줄', len(no_country), f'(그중 주소 칸이 각주 표시나 \'-\'뿐인 줄 {footnote_addr.sum()})',
      '/ 금액 단위를 찾지 못한 기업', unit_missing or '없음')
subsidiaries.country.value_counts().head(15)

종속회사 19236 줄 / 933 개 기업
나라를 정하지 못한 줄 2739 (그중 주소 칸이 각주 표시나 '-'뿐인 줄 1455) / 금액 단위를 찾지 못한 기업 없음


country
대한민국     7332
         2739
미국       2478
중국       1357
베트남       600
일본        374
싱가포르      368
인도네시아     359
인도        271
독일        254
호주        232
홍콩        218
영국        203
멕시코       181
말레이시아     159
Name: count, dtype: int64

In [6]:
# ── 확인: 종속회사 예 ──
# 예: 삼성전자의 미국 종속회사
subsidiaries[(subsidiaries.stock_code == '005930') & (subsidiaries.country == '미국')][
    ['name', 'alias', 'business', 'total_assets', 'is_major']].head(10)

,name,alias,business,total_assets,is_major
0,"Samsung Electronics America, Inc.",SEA,전자제품 판매,5.296636e+13,True
1,"Samsung International, Inc.",SII,전자제품 생산,2.413940e+12,True
3,"Samsung Electronics Home Appliances America, LLC",SEHA,가전제품 생산,4.781640e+11,True
4,"Samsung Research America, Inc",SRA,R&D,1.388428e+12,True
5,SAMSUNG NEXT LLC,SNX,해외자회사 관리,4.304800e+11,True
6,SAMSUNG NEXT FUND LLC,SNXF,"신기술사업자, 벤처기업 투자",5.188910e+11,True
7,NeuroLogica Corp.,,의료기기 생산 및 판매,3.145040e+11,True
8,"Samsung Lennox HVAC North America, LLC",,에어컨공조 판매,1.804270e+11,True
9,"Joyent, Inc.",,클라우드서비스,2.419950e+11,True
10,"SmartThings, Inc.",,스마트홈 플랫폼,2.512780e+11,True


## 5. 관계기업·공동기업 (`associates.csv`)

연결재무제표 주석의 "관계기업 및 공동기업 투자"에서 뽑는다. 이 주석은 공시 서식이 정해져 있지 않아 **회사마다 표 모양이 다르다.** 그래서 아래 규칙으로 표를 찾고 열을 고른다.

**주석 구간**: 원문의 "N. 연결재무제표 주석"부터 다음 장까지. 연결재무제표가 없는 기업(종속회사 없음)은 "N. 재무제표 주석"을 쓴다(`basis` = `연결`/`별도`).

**표 찾기**

1. 주석 제목에 "관계기업", "공동기업", "지분법", "관계회사", "지분증권"이 들어간 주석의 표 중 **지분율 열**이 있는 표를 쓴다.
2. 그런 표가 없으면, 다른 주석(주로 "1. 일반사항")의 표 중 "관계기업"·"공동기업" 구분 아래에 있는 줄만 쓴다. 같은 표에 종속기업이 섞여 있는 경우가 많기 때문이다.
3. 바로 앞 문단이 전기(작년)만 말하는 표("전기말 현재 …")는 쓰지 않는다.

**열 고르기**: 열 제목으로 정한다. 띄어쓰기는 무시한다.

| 열 | 열 제목 예 |
|---|---|
| 이름 | 회사명, 기업명, 피투자회사, 종목, 법인명. 제목이 비어 있으면 값이 숫자·구분이 아닌 앞쪽 열 |
| 소재지 | 소재지, 소재국가, 주사업장, 설립국가 |
| 주요 영업 | 주요 영업활동, 업종, 관계의 성격 |
| 지분율 | 지분율, 소유지분율. 당기말·전기말이 함께 있으면 당기말 |
| 구분 | 값이 "관계기업", "공동기업", "종속기업"인 열 |

**표준 주석 형식** (DART 재무 태그를 붙인 기업, 대상 기업 중 약 700개): 형식이 두 가지 더 있다.

- 열 제목이 비어 있고 지분율이 `0.1677`처럼 비율이다. 표의 지분율이 모두 1 이하이면 100을 곱한다.
- 회사가 열로, 항목(명칭·지분율·소재지)이 행으로 놓인 표다. 뒤집어서 읽는다. 이름이 열 머리글에만 있으면(`전체 관계기업 투자금액 관계기업 삼척블루파워㈜ 장부금액`) 머리글에서 이름을 꺼낸다.

**남기지 않는 줄**: 합계·소계, "기타", "<당기말>" 같은 구분 줄, 지분율도 소재지도 없는 줄. 같은 회사가 여러 표에 나오면 한 줄로 합치고 빈 칸만 채운다.

**한계**: 중요하지 않은 관계기업을 "기타"로 묶어 이름을 적지 않는 기업이 있다. 그런 회사는 이 표에 없고, 6절 타법인 출자현황에는 나온다(9절에서 겹치는 정도를 확인한다).

In [7]:
# ── 5. 관계기업: 원문에서 연결(없으면 별도)재무제표 주석 구간 찾기, 표 펼치기 ──
TITLE_RE = re.compile(r'<TITLE[^>]*>(.*?)</TITLE>', re.S)


def note_section(xml):
    """주석 구간을 돌려준다. (구간 XML, '연결' 또는 '별도')"""
    titles = [(m.start(), clean(re.sub(r'<[^>]+>', '', m.group(1)))) for m in TITLE_RE.finditer(xml)]

    def cut(pattern):
        starts = [p for p, t in titles if re.match(pattern, t)]
        if not starts:
            return ''
        start = starts[0]
        # 다음 장(예: "4. 재무제표")에서 끝난다. 주석 안의 제목("1. 일반사항 (연결)")은 끝으로 보지 않는다.
        ends = [p for p, t in titles if p > start and re.match(r'^\d+\.\s', t)
                and not re.search(r'\(연결\)|\(별도\)', t) and not re.match(r'^\d+\.\s*(연결)?재무제표\s*주석', t)]
        return xml[start:ends[0]] if ends else xml[start:]

    consolidated = cut(r'^\d+\.\s*연결재무제표\s*주석')
    if len(re.sub(r'<[^>]+>|\s', '', consolidated)) > 2000:     # 연결재무제표가 없으면 "해당사항 없음" 정도만 있다
        return consolidated, '연결'
    return cut(r'^\d+\.\s*재무제표\s*주석'), '별도'


def table_grid(table):
    """표를 rowspan·colspan을 펼친 칸 목록으로 바꾼다. [(머리글 줄인지, [칸, ...]), ...]"""
    out, pending = [], {}      # pending: 위 줄에서 rowspan으로 내려오는 칸 {열 번호: (값, 남은 줄 수)}
    for tr in table.find_all('tr'):
        row, col = [], 0
        cells = tr.find_all(['th', 'td', 'te', 'tu'], recursive=False)

        def fill_pending():
            nonlocal col
            while col in pending:
                text, left = pending[col]
                row.append(text)
                if left <= 1:
                    del pending[col]
                else:
                    pending[col] = (text, left - 1)
                col += 1

        for c in cells:
            fill_pending()
            text = clean(c.get_text(' '))
            colspan = int(c['colspan']) if str(c.get('colspan', '')).isdigit() else 1
            rowspan = int(c['rowspan']) if str(c.get('rowspan', '')).isdigit() else 1
            for _ in range(colspan):
                row.append(text)
                if rowspan > 1:
                    pending[col] = (text, rowspan - 1)
                col += 1
        fill_pending()
        is_head = bool(tr.find_parent('thead')) or bool(cells) and all(c.name == 'th' for c in cells)
        if any(row):
            out.append((is_head, row))
    return out

In [8]:
# ── 5. 관계기업: 표를 읽는 규칙(숫자, 지분율, 구분 값, 열 제목) ──
NUM_RE = re.compile(r'^[\(\-]?[\d,]+(\.\d+)?\)?%?$')
COL_RES = {
    'name': re.compile(r'회사명|기업명|법인명|피투자|종목|투자처|상호|명칭|회사|^기업$|관계기업명|공동기업명|투자회사'),
    'country': re.compile(r'소재|국가|주사업장|주된사업장|지역|설립국|국적|영업장소|사업장'),
    'business': re.compile(r'영업활동|주요사업|업종|사업내용|주요영업|활동|성격|업태|사업의내용|주된사업'),
    'ratio': re.compile(r'지분율|소유지분|持分|보유비율|지분비율|지분\(%\)|소유비율|투자비율'),
}
ASSOC_NOTE_RE = re.compile(r'관계기업|공동기업|지분법|관계회사|지분증권|공동약정')
TOTAL_RE = re.compile(r'^(합계|소계|총계|계|합계\(.*\))$')
NOT_NAME_RE = re.compile(r'^(피투자)?(회사|기업|법인|종목)명?$|^구분$|^명칭$|^<.*>$|^(당기|전기)말?$|^기타$|^기타\s|\s기타$|합계|소계'
                         r'|당기손익|공정가치측정|종속회사의\s*관계기업')
UNIT_ROW_RE = re.compile(r'\(?\s*단위\s*[:：].*\)?')


def is_num(text):
    """각주를 지우면 숫자(또는 -)뿐인 칸인가."""
    s = FOOTNOTE_RE.sub('', text).strip()
    return bool(NUM_RE.match(s)) or s in ('-', '－')


def to_ratio(text):
    """'23.7', '23.7%', '(*1) 30.0' → float. 0 초과 100 이하가 아니면 None"""
    v = to_num(FOOTNOTE_RE.sub('', text or '').replace('%', ''))
    return v if v is not None and 0 < v <= 100 else None


def category(text):
    """'<관계기업>', '전체 관계기업 투자금액', '종속기업:' 같은 구분 값을 '관계기업'·'공동기업'·'관계기업·공동기업'·'종속기업'·'기타'로. 구분이 아니면 ''"""
    t = re.sub(r'[\s<>\[\]():：·\-]|\d+\.|^[가-하]\.', '', text or '')
    t = re.sub(r'^전체', '', t)
    t = re.sub(r'(투자금액|투자주식|투자자산|투자|주식|현황|합계|등)$', '', t)
    if re.fullmatch(r'관계기업(및|과|,)?공동기업|공동기업(및|과|,)?관계기업', t):
        return '관계기업·공동기업'
    if re.fullmatch(r'관계기업|관계회사', t):
        return '관계기업'
    if re.fullmatch(r'공동기업|공동약정|조인트벤처|공동지배기업', t):
        return '공동기업'
    if re.fullmatch(r'(연결)?종속기업|종속회사|연결대상종속기업|지배기업', t):
        return '종속기업'
    if re.fullmatch(r'기타특수관계자|기타', t):
        return '기타'
    return ''


def kind_from_text(text):
    """문단·주석 제목에서 관계기업/공동기업을 읽는다. 예: '(2) 공동기업 투자' → '공동기업'"""
    t = re.sub(r'[()\s]', '', text or '')
    a, j = '관계기업' in t, ('공동기업' in t or '공동약정' in t)
    return '관계기업·공동기업' if a and j else '관계기업' if a else '공동기업' if j else ''


def header_count(rows):
    """앞쪽 머리글 줄 수. thead·th 줄이 없으면 숫자 칸이 없는 첫 줄을 머리글로 본다."""
    n = 0
    for is_head, r in rows:
        if is_head or (n == 0 and not any(is_num(c) for c in r[1:] if c)):
            n += 1
        else:
            break
    return max(n, 1)


def map_columns(labels):
    """열 제목으로 이름·소재지·주요 영업·지분율 열 번호를 정한다."""
    cols = {}
    for i, label in enumerate(labels):
        lab = label.replace(' ', '')
        for key, pattern in COL_RES.items():
            if key in cols and key != 'ratio':
                continue
            if not pattern.search(lab):
                continue
            if key == 'ratio' and 'ratio' in cols and ('전기' in lab or '당기' not in lab):
                continue                  # 지분율 열이 여럿이면 당기말 것을 쓴다
            if key == 'name' and ('ratio' in cols or 'country' in cols) and i > 2:
                continue
            if key == 'country' and COL_RES['ratio'].search(lab):
                continue
            cols[key] = i
            break
    return cols


def name_from_header(label):
    """'전체 관계기업 투자금액 관계기업 삼척블루파워㈜ 장부금액 총장부금액' → '삼척블루파워㈜'"""
    m = re.search(r'.*(?:관계\s*기업|공동\s*기업)\s+(.+)$', label or '')
    if not m:
        return ''
    name = re.sub(r'\s+(장부금액|총장부금액|합계|취득원가|공정가치|투자금액|당기|전기|기말|기초).*$', '', m.group(1)).strip()
    return '' if category(name) or name == '투자금액' else name

In [9]:
# ── 5. 관계기업: 주석 구간의 표를 문서 순서대로 읽기 ──
def read_table(rows):
    """표 한 개를 {'labels', 'cols', 'body', 'transposed'}로 정리한다."""
    while len(rows) > 2 and all(not c or UNIT_ROW_RE.fullmatch(c) for c in rows[0][1]):
        rows = [(True, r) if k == 0 else (h, r) for k, (h, r) in enumerate(rows[1:])]   # '(단위:천원)' 줄은 버린다
    n_head = header_count(rows)
    n_col = max(len(r) for _, r in rows)
    head = [' '.join(dict.fromkeys(r[i] for _, r in rows[:n_head] if i < len(r))) for i in range(n_col)]
    body = [r for _, r in rows[n_head:]]

    # 회사가 열, 항목이 행인 표(표준 주석 형식)는 뒤집는다
    first = [r[0].replace(' ', '') for r in body]
    has_name_row = any(re.search(r'명칭|기업명|회사명', f) for f in first)
    head_names = [name_from_header(h) for h in head[1:]]
    transposed = (any(COL_RES['ratio'].search(f) for f in first)
                  and (has_name_row or sum(bool(h) for h in head_names) >= max(1, (n_col - 1) // 2)))
    if transposed:
        labels = first + ['구분']
        body = [[r[j] if j < len(r) else '' for r in body] + [head[j]] for j in range(1, n_col)]
        cols = {'kind': len(labels) - 1}
        for i, lab in enumerate(first):
            for key, pattern in [('name', re.compile(r'명칭|기업명|회사명'))] + [(k, COL_RES[k]) for k in ('ratio', 'country', 'business')]:
                if key not in cols and pattern.search(lab):
                    cols[key] = i
                    break
        if 'name' not in cols:                         # 이름이 열 머리글에만 있는 표
            labels.append('명칭')
            for r, h in zip(body, head_names):
                r.append(h)
            cols['name'] = len(labels) - 1
        return {'labels': labels, 'cols': cols, 'body': body, 'transposed': True}

    cols = map_columns(head)
    if 'name' not in cols:                             # 이름 열 제목이 비어 있는 표
        first_mapped = min(cols.values(), default=n_col)
        cols['name'] = 0
        for i in range(first_mapped - 1, -1, -1):
            vals = [r[i] for r in body if i < len(r) and r[i]]
            if vals and sum(not category(v) and not is_num(v) for v in vals) >= 0.5 * len(vals):
                cols['name'] = i
                break
    for i in range(n_col):                             # 구분 열: 값이 대부분 '관계기업', '종속기업' 같은 구분인 열
        if i in cols.values():
            continue
        vals = [r[i] for r in body if i < len(r) and r[i]]
        if vals and sum(bool(category(v)) for v in vals) >= 0.5 * len(vals):
            cols['kind'] = i
            break
    return {'labels': head, 'cols': cols, 'body': body, 'transposed': False}


def note_tables(section):
    """주석 구간의 표를 문서 순서대로 읽고, 표마다 속한 주석 제목과 바로 앞 문단을 붙인다."""
    soup = BeautifulSoup(section, 'lxml')
    note_no, note_title, before, last_p = 0, '', '', ''
    tables = []
    for el in soup.find_all(['title', 'p', 'table']):
        if el.name in ('title', 'p'):
            text = clean(el.get_text(' '))
            if not text:
                continue
            m = re.match(r'^(\d{1,2})\s*\.\s*(\D.{0,80})$', text)
            # 주석 번호는 차례로 커진다. 주석 안의 번호 목록("1. …")에 속지 않도록 3 이내로 커질 때만 새 주석으로 본다.
            if m and (el.name == 'title' or note_no < int(m.group(1)) <= note_no + 3):
                note_no, note_title = int(m.group(1)), m.group(2)
            before, last_p = (before + ' ' + text)[-200:], text
            continue
        if el.find('table'):                           # 표를 감싼 바깥 표는 건너뛰고 안쪽 표만 읽는다
            continue
        grid = table_grid(el)
        if len(grid) >= 2:
            t = read_table(grid)
            t.update(note_no=note_no, note_title=note_title, before=before, last_p=last_p,
                     in_note=bool(ASSOC_NOTE_RE.search(re.sub(r'[()\s]', '', note_title))),
                     past_only=bool(re.search(r'전기', before)) and not re.search(r'당기', before))
            tables.append(t)
        before, last_p = '', ''
    return tables

In [10]:
# ── 5. 관계기업: 표에서 관계기업 줄 뽑기 ──
def table_associates(t, need_category):
    """표 한 개에서 관계기업 줄을 뽑는다. need_category면 '관계기업'·'공동기업' 구분 아래 줄만 쓴다."""
    cols = t['cols']
    ratios = [to_ratio(r[cols['ratio']]) for r in t['body'] if 'ratio' in cols and cols['ratio'] < len(r)]
    fraction = any(v is not None for v in ratios) and max(v for v in ratios if v is not None) <= 1.0   # 0.1677 형식
    kind = kind_from_text(t['last_p']) or kind_from_text(t['before']) or (kind_from_text(t['note_title']) if t['in_note'] else '')
    current = None if need_category else kind
    prev_country, out = '', []
    for r in t['body']:
        get = lambda k: r[cols[k]] if k in cols and cols[k] < len(r) else ''
        name = get('name')
        cat = category(name)
        if cat and len(set(c for c in r if c)) <= 2 and not to_ratio(get('ratio')):   # '<관계기업>' 같은 구분 줄
            current = None if cat in ('종속기업', '기타') else cat
            continue
        row_kind = current
        if 'kind' in cols:
            k = category(get('kind')) or (kind_from_text(get('kind')) if t['transposed'] else '')
            if k in ('종속기업', '기타'):
                continue
            row_kind = k or current
        if need_category and not row_kind:
            continue
        compact = name.replace(' ', '')
        if not name or cat or is_num(name) or TOTAL_RE.match(compact) or NOT_NAME_RE.search(compact):
            continue
        ratio = to_ratio(get('ratio'))
        if ratio is not None and fraction:
            ratio = round(ratio * 100, 2)
        place = get('country')
        if place in ('〃', '"', '″', '상동', '동일'):
            place = prev_country
        prev_country = place or prev_country
        if ratio is None and not place:
            continue
        out.append({'name_raw': name, 'place': place, 'business': get('business'), 'ratio': ratio,
                    'kind': row_kind or '', 'note': f"{t['note_no']}. {t['note_title']}"})
    return out


def find_associates(xml):
    """원문 하나에서 관계기업 목록을 뽑는다. (줄 목록, '연결'/'별도')"""
    section, basis = note_section(xml)
    tables = [t for t in note_tables(section)
              if 'ratio' in t['cols'] and not t['past_only']
              and (t['in_note'] or 'country' in t['cols'] or 'business' in t['cols'])]
    found = [r for t in tables if t['in_note'] for r in table_associates(t, need_category=False)]
    if not found:      # 관계기업 주석에 목록 표가 없으면 일반사항 등의 표에서 '관계기업' 구분 아래 줄을 쓴다
        for t in tables:
            if t['in_note']:
                continue
            cats = {category(r[t['cols'][k]]) for r in t['body'] for k in ('name', 'kind') if k in t['cols'] and t['cols'][k] < len(r)}
            if cats & {'관계기업', '공동기업', '관계기업·공동기업'} or re.search(r'관계\s*기업|공동\s*기업', t['before']):
                found += table_associates(t, need_category=bool(cats - {''}))
    merged = {}
    for r in found:          # 같은 회사가 여러 표에 나오면 한 줄로 합치고 빈 칸만 채운다
        key = clean_name(r['name_raw'])
        if key not in merged:
            merged[key] = r
        else:
            for k, v in r.items():
                if v not in ('', None) and merged[key].get(k) in ('', None):
                    merged[key][k] = v
    return list(merged.values()), basis

In [11]:
# ── 5. 관계기업: 원문 1,063개를 읽어 associates.csv로(이 노트북 시간의 대부분) ──
rows, meta = [], []
for i, d in enumerate(docs.itertuples(), start=1):
    found, basis = find_associates(main_xml(RAW / d.file))
    meta.append({'stock_code': d.stock_code, 'basis': basis, 'n': len(found)})
    for r in found:
        name, alias = split_alias(clean_name(r['name_raw']))
        rows.append({
            'stock_code': d.stock_code, 'corp_name': CORP_NAME.get(d.stock_code, d.corp_name),
            'name': name, 'alias': alias, 'name_raw': r['name_raw'], 'kind': r['kind'],
            'country': country_of(r['place']), 'place': r['place'], 'business': r['business'], 'ratio': r['ratio'],
            'note': r['note'], 'basis': basis,
            'rcept_no': d.rcept_no, 'report': d.report, 'base_date': d.base_date,
        })
    if i % 200 == 0:
        print(f'  {i}/{len(docs)}')

associates = pd.DataFrame(rows)
associates.to_csv(OUT / 'associates.csv', index=False, encoding='utf-8-sig')
assoc_meta = pd.DataFrame(meta)
print('관계기업·공동기업', len(associates), '줄 /', associates.stock_code.nunique(), '개 기업')
print('주석 구간:', assoc_meta.basis.value_counts().to_dict(), '/ 0개로 나온 기업', (assoc_meta.n == 0).sum())
print('구분:', associates.kind.replace('', '(없음)').value_counts().to_dict())
print('지분율 빈칸', associates.ratio.isna().sum(), '/ 나라 빈칸', (associates.country == '').sum())
associates.country.value_counts().head(15)

  200/1063


  400/1063


  600/1063


  800/1063


  1000/1063


관계기업·공동기업 8357 줄 / 704 개 기업
주석 구간: {'연결': 933, '별도': 130} / 0개로 나온 기업 359
구분: {'관계기업': 5678, '관계기업·공동기업': 1911, '공동기업': 742, '(없음)': 26}
지분율 빈칸 358 / 나라 빈칸 1364


country
대한민국       5482
           1364
미국          340
중국          251
베트남          99
싱가포르         78
인도네시아        74
일본           56
인도           45
태국           37
사우디아라비아      31
호주           31
홍콩           29
케이맨제도        28
말레이시아        27
Name: count, dtype: int64

In [12]:
# ── 확인: 관계기업 예 ──
# 예: 미국 뉴스와 연결될 해외 합작사·관계기업
associates[associates.stock_code.isin(['005380', '000660', '035420'])][
    ['corp_name', 'name', 'alias', 'kind', 'country', 'business', 'ratio']].head(20)

,corp_name,name,alias,kind,country,business,ratio
7,SK하이닉스,SK China Company Limited,,관계기업,중국,컨설팅 및 투자,11.87
8,SK하이닉스,SK South East Asia Investment Pte. Ltd.,,관계기업,싱가포르,컨설팅 및 투자,20.00
9,SK하이닉스,"SiFive, Inc.",,관계기업,미국,반도체 설계 및 제조,6.84
10,SK하이닉스,우시시신파직접회로산업원유한공사,,관계기업,중국,과학기술단지 개발,30.00
11,SK하이닉스,"HITECH Semiconductor (Wuxi) Co., Ltd.",,공동기업,중국,반도체 부품 제조,45.00
12,SK하이닉스,"SK hynix system ic (Wuxi) Co., Ltd.",,공동기업,중국,파운드리사업,49.79
13,SK하이닉스,반도체성장 전문투자형 사모 투자신탁,,공동기업,대한민국,투자,33.33
14,SK하이닉스,시스템반도체 상생 전문투자형 사모투자신탁,,공동기업,대한민국,투자,37.50
41,현대자동차,북경현대기차금융유한공사,BHAF,관계기업·공동기업,중국,금융,53.00
42,현대자동차,산동현대위아기차발동기유한공사,WAE,관계기업·공동기업,중국,자동차부품 제조 및 판매,31.40


## 6. 타법인 출자현황 (`investments.csv`)

API 응답을 그대로 표로 옮긴다. 모든 출자 대상이 들어 있다(단순투자 포함).

| 열 | API 항목 | 뜻 |
|---|---|---|
| `purpose` | `invstmnt_purps` | 출자 목적. 회사가 자유롭게 적은 글 그대로다(경영참여, 경영참가, 단순투자, 일반투자, 투자 등) |
| `purpose_group` | `purpose`에서 만듦 | 경영참여 / 단순투자 / 일반투자 / 기타 |
| `ratio` | `trmend_blce_qota_rt` | 기말 지분율(%). 0~100을 벗어나면 빈칸 |
| `book_value` | `trmend_blce_acntbk_amount` | 기말 장부가액(원) |
| `first_acq_date` | `frst_acqs_de` | 최초 취득일 |
| `listed` | 이름의 `(상장)`/`(비상장)` 표시 | 표시한 기업만 채운다 |
| `target_assets`, `target_net_income` | `recent_bsns_year_fnnr_sttus_*` | 출자 대상의 최근 사업연도 자산총계·당기순손익(원) |

- 합계 줄과 이름이 `-`인 줄은 뺀다.
- 출자 목적이 `"`, `〃`(위와 같음)이면 바로 위 줄의 출자 목적을 쓴다.
- `purpose_group`: "단순"이 들어가거나 `단투`이면 단순투자, "경영"·`경참`·"지배"·"계열"·"합작"·`JV`가 들어가면 경영참여, "일반"이 들어가거나 `일투`이면 일반투자, 나머지(`투자`, `-` 등)는 기타다. **그래프에서는 단순투자를 뺀다**(graphDB.md 3.2). 출자 목적은 회사가 적는 값이라 틀린 경우도 있다(현대건설은 지분 100% 해외 자회사를 단순투자로 적었다). 그런 회사는 종속회사 표에 들어 있어 그래프에서 빠지지 않는다.

In [13]:
# ── 6. 타법인 출자현황(investments.csv): API 응답을 그대로 표로 ──
def api_rows(item):
    """selected_reports.csv에서 항목 하나의 API 파일을 읽어 (보고서 줄, 응답 줄) 쌍으로 돌려준다."""
    for s in selected[selected['item'] == item].itertuples():
        for e in read_saved(RAW / s.file)['response'].get('list', []):
            yield s, e


SAME_PURPOSE = ('"', '〃', '″', '상동', '동일')


def purpose_group(purpose):
    """출자 목적 글을 단순투자 / 경영참여 / 일반투자 / 기타로 묶는다(약칭 "단투"·"경참" 포함)."""
    p = purpose.replace(' ', '')
    if '단순' in p or p == '단투':
        return '단순투자'
    if re.search(r'경영|경참|지배|계열|합작|JV', p):
        return '경영참여'
    if '일반' in p or p == '일투':
        return '일반투자'
    return '기타'


rows, prev = [], {}
for s, e in api_rows('타법인 출자현황'):
    purpose = clean(e.get('invstmnt_purps'))
    if purpose in SAME_PURPOSE:                       # '위와 같음' 표시는 바로 위 줄의 출자 목적으로
        purpose = prev.get(s.file, purpose)
    prev[s.file] = purpose
    raw_name = clean(e.get('inv_prm'))
    if raw_name in ('', '-', '합계', '합 계', '계', '소계'):
        continue
    listed = re.search(r'\((비?상장)\)', raw_name)
    name, alias = split_alias(clean_name(raw_name))
    rows.append({
        'stock_code': s.stock_code, 'corp_name': CORP_NAME.get(s.stock_code, s.corp_name),
        'name': name, 'alias': alias, 'name_raw': e.get('inv_prm', ''),
        'purpose': purpose, 'purpose_group': purpose_group(purpose), 'ratio_raw': clean(e.get('trmend_blce_qota_rt')),
        'ratio': valid_ratio(e.get('trmend_blce_qota_rt')), 'shares': to_num(e.get('trmend_blce_qy')),
        'book_value': to_num(e.get('trmend_blce_acntbk_amount')), 'first_acq_date': clean(e.get('frst_acqs_de')),
        'listed': listed.group(1) if listed else '',
        'target_assets': to_num(e.get('recent_bsns_year_fnnr_sttus_tot_assets')),
        'target_net_income': to_num(e.get('recent_bsns_year_fnnr_sttus_thstrm_ntpf')),
        'rcept_no': e['rcept_no'], 'report': s.report, 'base_date': e.get('stlm_dt', ''),
    })
investments = pd.DataFrame(rows)
investments.to_csv(OUT / 'investments.csv', index=False, encoding='utf-8-sig')
print('타법인 출자', len(investments), '줄 /', investments.stock_code.nunique(), '개 기업')
print('기준일:', investments.base_date.value_counts().head(6).to_dict())
bad = investments[investments.ratio.isna() & (investments.ratio_raw.str.replace(',', '').str.fullmatch(r'-?[\d.]+'))]
print('지분율이 0~100을 벗어나 빈칸으로 둔 줄:', bad[['corp_name', 'name', 'ratio_raw']].values.tolist())
investments.purpose_group.value_counts()

타법인 출자 19936 줄 / 1007 개 기업
기준일: {'2026-06-30': 19785, '2026-09-30': 70, '2026-03-31': 52, '2026-05-31': 15, '2025-12-31': 14}
지분율이 0~100을 벗어나 빈칸으로 둔 줄: [['대상홀딩스', 'PT DAESANG AGUNG INDONESIA', '204'], ['코스맥스비티아이', '휴앤바이옴', '1000'], ['컴투스', '(주)케이뱅크', '6440308']]


purpose_group
단순투자    7221
경영참여    7147
기타      2823
일반투자    2745
Name: count, dtype: int64

## 7. 최대주주 현황 (`shareholders.csv`)

최대주주와 특수관계인이 한 줄씩이다. 개인(임원, 총수 일가)도 들어 있다. 그래프의 모회사 관계에는 법인만 쓰므로 `is_corp`으로 법인 여부를 표시한다.

- `is_corp`: 이름에 법인 표시(㈜, 주식회사, 재단, 은행, Inc, Ltd, LLC 등)가 있거나 영문 이름이면 법인으로 본다. 한글 이름 2~4자는 개인으로 본다. 규칙으로 정한 값이라 틀릴 수 있다.
- 같은 주주가 보통주·우선주로 두 줄일 수 있다(`stock_kind`).
- 합계(`계`) 줄은 뺀다.

In [14]:
# ── 7. 최대주주 현황(shareholders.csv): 법인 주주 여부(is_corp)를 붙인다 ──
CORP_MARK_RE = re.compile(r'㈜|\(주\)|\(유\)|\(재\)|\(사\)|주식회사|유한회사|유한책임회사|재단|법인|공단|공사|조합|협회|은행|증권|보험|'
                          r'자산운용|투자|홀딩스|그룹|펀드|신탁|연기금|국민연금|기금|정부|[A-Za-z]{2,}')

rows = []
for s, e in api_rows('최대주주 현황'):
    raw_name = clean(e.get('nm'))
    if raw_name in ('', '-', '계', '합계', '합 계', '소계'):
        continue
    name = clean_name(raw_name)
    is_corp = bool(CORP_MARK_RE.search(name)) and not re.fullmatch(r'[가-힣]{2,4}', name)
    rows.append({
        'stock_code': s.stock_code, 'corp_name': CORP_NAME.get(s.stock_code, s.corp_name),
        'name': name, 'name_raw': e.get('nm', ''), 'relation': clean(e.get('relate')), 'is_corp': is_corp,
        'stock_kind': clean(e.get('stock_knd')), 'ratio': valid_ratio(e.get('trmend_posesn_stock_qota_rt')),
        'shares': to_num(e.get('trmend_posesn_stock_co')), 'note': clean(e.get('rm')),
        'rcept_no': e['rcept_no'], 'report': s.report, 'base_date': e.get('stlm_dt', ''),
    })
shareholders = pd.DataFrame(rows)
shareholders.to_csv(OUT / 'shareholders.csv', index=False, encoding='utf-8-sig')
print('최대주주·특수관계인', len(shareholders), '줄 /', shareholders.stock_code.nunique(), '개 기업 / 법인',
      shareholders.is_corp.sum(), '줄')
# 예: 영문 이름 법인 주주 (해외 모회사 후보)
shareholders[shareholders.is_corp & shareholders.name.str.contains('[A-Za-z]{3,}', regex=True)
             & (shareholders.ratio >= 20)][['corp_name', 'name', 'relation', 'ratio', 'base_date']].head(15)

최대주주·특수관계인 9101 줄 / 1064 개 기업 / 법인 1603 줄


,corp_name,name,relation,ratio,base_date
485,S-Oil,Aramco Overseas Company B.V.,최대주주 본인,63.41,2026-06-30
1305,에스원,"SECOM CO., LTD.",본인,25.65,2026-06-30
2317,OCI,OCI 홀딩스㈜,최대주주,44.97,2026-06-30
2705,NICE평가정보,(주)NICE홀딩스,특수관계인,44.31,2026-06-30
2760,유안타증권,Yuanta Securities Asia Financial Services Priv...,최대주주 본인,61.09,2026-06-30
2768,SNT다이내믹스,(주)SNT홀딩스,최대주주,42.27,2026-06-30
2790,한국쉘석유,Shell Petroleum B.V,투자법인,53.85,2026-06-30
2881,SNT모티브,(주)SNT홀딩스,최대주주,45.12,2026-06-30
3041,SNT에너지,(주)SNT홀딩스,최대주주,47.97,2026-06-30
3393,한국철강,KISCO홀딩스(주),본인,60.34,2026-06-30


## 8. 수시공시: 양수·양도, 합병·분할 결정 (`decisions.csv`)

정기보고서 사이에 생긴 지분·구조 변화다. API마다 항목 이름이 달라 아래처럼 같은 열로 맞춘다.

| 종류 | API | `target` (대상 회사) | `counterparty` (거래 상대) | `ratio_after` |
|---|---|---|---|---|
| 타법인주식 양수 | `otcprStkInvscrInhDecsn` | 주식을 사는 회사 `iscmp_*` | 파는 쪽 `dlptn_*` | 양수 후 지분율 `atinh_eqrt` |
| 타법인주식 양도 | `otcprStkInvscrTrfDecsn` | 주식을 파는 회사 `iscmp_*` | 사는 쪽 `dlptn_*` | 양도 후 지분율 `attrf_eqrt` |
| 합병 | `cmpMgDecsn` | 합병 상대 `mgptncmp_*` | | |
| 분할 | `cmpDvDecsn` | 분할 신설회사 `dvfcmp_*` | | |
| 분할합병 | `cmpDvmgDecsn` | 합병 상대 `mgptncmp_*` | 분할 신설회사 `dvfcmp_*` | |

- `rcept_date`는 접수번호 앞 8자리(공시 제출일), `board_date`는 이사회 결의일이다.
- `after_latest_report`: 이 공시가 그 기업의 최신 사업보고서 기준일 뒤에 나왔는지. 참이면 종속회사·관계기업 표에 아직 반영되지 않은 변화일 수 있다.
- `detail`: 합병·분할 방법 설명 앞부분(200자).

In [15]:
# ── 8. 수시공시(decisions.csv): 양수·양도, 합병·분할 결정 ──
DECISION_APIS = {
    'otcprStkInvscrInhDecsn': ('타법인주식 양수', 'iscmp', 'dlptn', 'atinh_eqrt', 'inh_pp'),
    'otcprStkInvscrTrfDecsn': ('타법인주식 양도', 'iscmp', 'dlptn', 'attrf_eqrt', 'trf_pp'),
    'cmpMgDecsn': ('합병', 'mgptncmp', None, None, 'mg_mth'),
    'cmpDvDecsn': ('분할', 'dvfcmp', None, None, 'dv_mth'),
    'cmpDvmgDecsn': ('분할합병', 'mgptncmp', 'dvfcmp', None, 'dvmg_mth'),
}
annual_base = dict(zip(docs.stock_code, docs.base_date))    # 최신 사업보고서 기준일


def value(e, key):
    """API 항목 값을 정리한다. '-'·'해당사항 없음'은 빈칸."""
    v = clean(e.get(key)) if key else ''
    return '' if v in ('-', '해당사항없음', '해당사항 없음') else v


rows = []
for api, (kind, target, counter, ratio_key, detail_key) in DECISION_APIS.items():
    for path in sorted((RAW / 'decisions' / api).glob('*.json')):
        stock_code = path.stem.split('_')[0]
        for e in read_saved(path)['response'].get('list', []):
            rcept_date = f"{e['rcept_no'][:4]}-{e['rcept_no'][4:6]}-{e['rcept_no'][6:8]}"
            rows.append({
                'stock_code': stock_code, 'corp_name': CORP_NAME.get(stock_code, e.get('corp_name', '')), 'kind': kind,
                'target': clean_name(value(e, f'{target}_cmpnm')),
                'target_nation': value(e, f'{target}_nt') if target == 'iscmp' else '',
                'target_business': value(e, f'{target}_mbsn'),
                'target_relation': value(e, f'{target}_rl_cmpn'),
                'counterparty': clean_name(value(e, f'{counter}_cmpnm')) if counter else '',
                'counterparty_relation': value(e, f'{counter}_rl_cmpn') if counter else '',
                'ratio_after': valid_ratio(value(e, ratio_key)) if ratio_key else None,
                'detail': value(e, detail_key)[:200],
                'board_date': value(e, 'bddd'), 'rcept_date': rcept_date, 'rcept_no': e['rcept_no'],
                'after_latest_report': bool(annual_base.get(stock_code)) and rcept_date > annual_base[stock_code],
            })
decisions = pd.DataFrame(rows).drop_duplicates('rcept_no').sort_values('rcept_no', ascending=False)
decisions.to_csv(OUT / 'decisions.csv', index=False, encoding='utf-8-sig')
print('결정 공시', len(decisions), '건 / 최신 사업보고서 이후', decisions.after_latest_report.sum(), '건')
print(decisions.kind.value_counts().to_dict())
decisions[['corp_name', 'kind', 'target', 'target_nation', 'target_business', 'ratio_after', 'rcept_date']].head(15)

결정 공시 175 건 / 최신 사업보고서 이후 97 건
{'합병': 100, '타법인주식 양수': 34, '분할': 22, '타법인주식 양도': 18, '분할합병': 1}


,corp_name,kind,target,target_nation,target_business,ratio_after,rcept_date
26,더블유게임즈,타법인주식 양수,"주식회사 더블다운인터액티브 (DoubleDown Interactive Co., Ltd.)",대한민국,게임 등 소프트웨어 개발 및 공급업,100.0,2026-11-10
27,삼성바이오로직스,타법인주식 양수,PolyPeptide Group AG,스위스 (Switzerland),펩타이드 CDMO,100.0,2026-11-06
155,삼부토건,분할,에스비자산관리 주식회사,,"토목건축공사업, 부동산 시행업 등",NaN,2026-11-05
164,카카오,분할,주식회사 카카오에이아이 (가칭) (KakaoAI Corp.) (가칭),,카카오톡 기반 플랫폼 사업,NaN,2026-11-05
102,카카오,합병,주식회사 카카오인베스트먼트,,경영컨설팅,NaN,2026-11-05
10,KG스틸,타법인주식 양수,케이지모빌리티플랫폼(주),대한민국,중고 자동차 판매업,52.5,2026-11-03
75,샘표,합병,샘표식품주식회사(SEMPIO FOODS COMPANY),,기타 식품 제조업,NaN,2026-10-30
136,샘표식품,합병,샘표주식회사(SEMPIO COMPANY),,지주회사,NaN,2026-10-30
149,SK아이이테크놀로지,합병,"SK이노베이션(주) (SK Innovation Co., Ltd.)",,석유 정제품 제조업,NaN,2026-10-28
55,DH오토넥스,합병,(주)대유플러스자산관리,,자동차 부품 제조업 등,NaN,2026-10-28


## 9. 결과 확인

**첫 번째 표**: 기업마다 표별 줄 수. `parse_summary.csv`로도 저장한다.

**두 번째 확인**: 관계기업 표가 제대로 뽑혔는지 타법인 출자현황과 맞춰 본다. 타법인 출자 중 지분율 20~50%(보통 관계기업이 되는 범위)인 대상이 관계기업 표나 종속회사 표에 있는지 본다. 이름 표기가 달라(`(주)`와 `㈜`, 약칭) 실제보다 낮게 나오므로, 기업별로 비율이 크게 낮은 곳을 직접 살펴보는 데 쓴다.

In [16]:
# ── 9. 결과 확인: 기업마다 표별 줄 수(parse_summary.csv) ──
summary = companies[['stock_code', 'stock_name', 'group']].copy()
for label, df in [('종속회사', subsidiaries), ('관계기업', associates), ('타법인출자', investments),
                  ('최대주주', shareholders), ('수시공시', decisions)]:
    summary[label] = summary.stock_code.map(df.groupby('stock_code').size()).fillna(0).astype(int)
summary['해외_종속회사'] = summary.stock_code.map(
    subsidiaries[~subsidiaries.country.isin(['대한민국', ''])].groupby('stock_code').size()).fillna(0).astype(int)
summary['해외_관계기업'] = summary.stock_code.map(
    associates[~associates.country.isin(['대한민국', ''])].groupby('stock_code').size()).fillna(0).astype(int)
summary['사업보고서'] = summary.stock_code.isin(docs.stock_code)
summary.to_csv(OUT / 'parse_summary.csv', index=False, encoding='utf-8-sig')

print('표별 줄 수:', {k: len(v) for k, v in [('companies', companies), ('subsidiaries', subsidiaries), ('associates', associates),
                                            ('investments', investments), ('shareholders', shareholders), ('decisions', decisions)]})
print('사업보고서가 없어 종속회사·관계기업을 뽑지 못한 기업:', summary[~summary['사업보고서']].stock_name.tolist())
print('타법인 출자·최대주주가 모두 없는 기업:', summary[(summary['타법인출자'] == 0) & (summary['최대주주'] == 0)].stock_name.tolist())
summary.head(10)

표별 줄 수: {'companies': 1067, 'subsidiaries': 19236, 'associates': 8357, 'investments': 19936, 'shareholders': 9101, 'decisions': 175}
사업보고서가 없어 종속회사·관계기업을 뽑지 못한 기업: ['한화머시너리앤서비스홀딩스', '엘리스', '마키나락스', '디티에스']
타법인 출자·최대주주가 모두 없는 기업: ['한화머시너리앤서비스홀딩스', '엘리스', '디티에스']


,stock_code,stock_name,group,종속회사,관계기업,타법인출자,최대주주,수시공시,해외_종속회사,해외_관계기업,사업보고서
0,005930,삼성전자,코스피200,308,7,127,23,0,271,0,True
1,000660,SK하이닉스,코스피200,54,8,44,11,0,40,6,True
2,402340,SK스퀘어,코스피200,16,16,16,12,0,6,3,True
3,373220,LG에너지솔루션,코스피200,25,6,40,1,0,24,2,True
4,009150,삼성전기,코스피200,14,2,22,6,0,11,0,True
5,207940,삼성바이오로직스,코스피200,1,2,3,6,2,1,0,True
6,005380,현대자동차,코스피200,164,20,215,18,0,124,10,True
7,105560,KB금융,코스피200,376,88,11,1,0,39,1,True
8,032830,삼성생명,코스피200,163,21,64,8,0,12,7,True
9,028260,삼성물산,코스피200,139,10,146,15,0,116,6,True


In [17]:
# ── 확인: 출자 대상이 관계기업·종속회사 표에서도 찾아지는지 ──
def name_key(name):
    """비교용 이름: 법인 표시·괄호·공백·문장부호를 지운 소문자"""
    s = re.sub(r'\((주|유|사|재)\)|㈜|주식회사|유한회사|\([^)]*\)|[\s.,\-·&\'"]', '', str(name).lower())
    return re.sub(r'(inc|ltd|llc|corp|corporation|coltd|limited|co)$', '', s)


def matches(key, pool):
    """이름 키가 목록에 있거나, 3자 이상끼리 한쪽이 다른 쪽에 들어 있으면 같은 회사로 본다."""
    return key in pool or any(len(key) >= 3 and len(p) >= 3 and (key in p or p in key) for p in pool)


pools = {}
for df, col in [(associates, 'assoc'), (subsidiaries, 'sub')]:
    for code, g in df.groupby('stock_code'):
        pools.setdefault(code, {})[col] = {name_key(n) for n in g['name']} | {name_key(a) for a in g['alias'] if a}

check = investments[(investments.ratio >= 20) & (investments.ratio < 50)].copy()
check['in_assoc'] = [matches(name_key(n), pools.get(c, {}).get('assoc', set())) for c, n in zip(check.stock_code, check.name)]
check['in_sub'] = [matches(name_key(n), pools.get(c, {}).get('sub', set())) for c, n in zip(check.stock_code, check.name)]
check['found'] = check.in_assoc | check.in_sub
print(f'타법인 출자 중 지분율 20~50%: {len(check)}줄')
print(f'  관계기업 표에 있음 {check.in_assoc.mean():.0%}, 종속회사 표에 있음 {check.in_sub.mean():.0%}, 둘 중 하나에 있음 {check.found.mean():.0%}')
by_company = check.groupby(['stock_code', 'corp_name']).agg(출자_20_50=('found', 'size'), 찾음=('found', 'sum')).reset_index()
by_company = by_company.merge(summary[['stock_code', '관계기업']], on='stock_code')
low = by_company[(by_company['출자_20_50'] >= 3) & (by_company['찾음'] < by_company['출자_20_50'] / 2)]
print(f'20~50% 출자 대상이 3개 이상인데 절반도 찾지 못한 기업: {len(low)}개')
low.sort_values('출자_20_50', ascending=False).head(15)

타법인 출자 중 지분율 20~50%: 2600줄
  관계기업 표에 있음 59%, 종속회사 표에 있음 16%, 둘 중 하나에 있음 74%
20~50% 출자 대상이 3개 이상인데 절반도 찾지 못한 기업: 50개


,stock_code,corp_name,출자_20_50,찾음,관계기업
180,006800,미래에셋증권,43,15,87
147,005380,현대자동차,40,17,20
360,035420,NAVER,29,13,45
13,000270,기아,27,9,29
32,000880,한화,24,10,2
255,013580,계룡건설산업,22,9,22
282,017670,SK텔레콤,20,7,18
330,030200,케이티,19,3,12
324,028260,삼성물산,19,8,10
40,001200,유진증권,15,3,3


In [18]:
# ── 확인: 찾지 못한 출자 대상 예 ──
# 찾지 못한 출자 대상 예 (이름 표기 차이, 주석에서 '기타'로 묶인 투자, 기준일 차이가 대부분이다)
check[~check.found][['corp_name', 'name', 'purpose', 'ratio', 'base_date']].head(15)

,corp_name,name,purpose,ratio,base_date
21,삼성전자,TSST Japan,경영참여,49.00,2026-06-30
35,삼성전자,㈜삼성글로벌리서치,경영참여,29.80,2026-06-30
159,SK하이닉스,미래에셋위반도체 제1호 창업벤처전문사모투자 합작회사,일반투자,30.00,2026-06-30
163,SK하이닉스,SK Telecom Japan,경영참여,25.30,2026-06-30
164,SK하이닉스,"SK Americas, Inc.",경영참여,20.00,2026-06-30
167,SK하이닉스,농업회사법인 푸르메소셜팜(주),경영참여,35.50,2026-06-30
196,LG에너지솔루션,이차전지 성장펀드,단순투자,33.00,2026-06-30
217,LG에너지솔루션,코리아기가플랫폼㈜,단순투자,20.00,2026-06-30
276,현대자동차,Beijing HYUNDAI TRANSYS Transmission,경영참여,24.08,2026-06-30
277,현대자동차,Hyundai Transys (Shandong),경영참여,30.00,2026-06-30


## 10. 계열회사 (`affiliates.csv`, `affiliate_groups.csv`)

사업보고서 원문 "IX. 계열회사 등에 관한 사항"에서 뽑는다. 그래프의 그룹 소속(`IN_GROUP`)에 쓴다. 이름 앞부분으로 그룹을 정하면 SK증권(2018년 SK에서 나감)·롯데손해보험처럼 틀리고, ISC(SK)·디엔에프(솔브레인)처럼 이름으로는 알 수 없는 계열사를 놓친다.

- **기업집단 이름**: IX절 요약표("기업집단의 명칭 / 계열회사의 수 / 상장 / 비상장")의 이름과 상장·비상장 회사 수. 이름이 "-"면 기업집단이 없다. 이름 표기는 회사마다 다르다(엘지, LG계열, (주)LG). 그래서 그래프는 이름이 아니라 계열회사 목록으로 같은 그룹을 묶는다.
- **계열회사 목록**: 상세표 "2. 계열회사 현황(상세)"의 상장 여부, 기업명, 법인등록번호.
- **상장 계열회사의 종목코드**(`affiliate_stock_code`): 법인등록번호(대상 기업의 기업개황) → 이름에 적힌 종목코드 → 이름(`sector.json` 종목명, DART 고유번호 목록의 법인명) 순서로 찾는다. 이름은 법인 표기를 지우고, 한글로 풀어 쓴 영문 약칭(엘지, 씨제이, 에스케이 등)을 영문으로 바꿔 비교한다.

| 표 | 한 줄 |
|---|---|
| `affiliates.csv` | 보고서 하나에 적힌 계열회사 하나: 보고 회사, 기업집단 이름, 계열회사 이름, 상장 여부, 법인등록번호, 종목코드, 접수번호·보고서·기준일 |
| `affiliate_groups.csv` | 보고 회사 하나: 기업집단 이름, 상장·비상장 계열회사 수 |

In [19]:
# ── 10. 계열회사(affiliates.csv): IX. 계열회사 요약표의 기업집단 이름, 상세표 2. 계열회사 현황(상세)의 계열회사 목록 ──
import xml.etree.ElementTree as ET

NONE_VALUES = {'', '-', '해당사항 없음', '해당사항없음', '해당없음'}


def group_summary(xml):
    """IX절 요약표에서 (기업집단 이름, 상장 수, 비상장 수)를 읽는다. 표가 없거나 이름이 '-'면 None."""
    titles = [(m.start(), clean(re.sub(r'<[^>]+>', '', m.group(1)))) for m in TITLE_RE.finditer(xml)]
    ix = [p for p, t in titles if re.match(r'^IX\.\s*계열회사', t)]
    if not ix:
        return None
    end = next((p for p, t in titles if p > ix[-1] and re.match(r'^X\.', t)), len(xml))
    for table in BeautifulSoup(xml[ix[-1]:end], 'lxml').find_all('table'):
        grid = [row for _, row in table_grid(table)]
        flat = ' '.join(' '.join(r) for r in grid)
        if '기업집단' not in flat or '비상장' not in flat:
            continue
        for row in grid:
            if len(row) >= 4 and not re.search(r'기업집단|상장|명칭', row[0]):
                name = clean(row[0])
                if name in NONE_VALUES:
                    return None
                return name, to_num(row[-3]), to_num(row[-2])
    return None


def affiliate_rows(xml):
    """상세표 2. 계열회사 현황(상세)의 줄을 (상장 여부, 기업명, 법인등록번호)로 돌려준다."""
    titles = [(m.start(), m.end(), clean(re.sub(r'<[^>]+>', '', m.group(1)))) for m in TITLE_RE.finditer(xml)]
    hit = [(s, e) for s, e, t in titles if re.search(r'계열회사\s*현황\s*\(상세\)', t)]
    if not hit:
        return []
    start = hit[-1][1]
    end = next((s for s, _, _ in titles if s > start), len(xml))
    out = []
    for table in BeautifulSoup(xml[start:end], 'lxml').find_all('table'):
        for _, row in table_grid(table):
            if len(row) < 3:
                continue
            kind = clean(row[0]).replace(' ', '')
            if kind not in ('상장', '비상장'):
                continue
            name, reg = clean(row[-2]), clean(row[-1])
            if name in NONE_VALUES or re.fullmatch(r'[\d,]+', name):
                continue
            out.append((kind, name, reg))
    return out


rows, groups = [], []
for d in docs.itertuples():
    xml = main_xml(RAW / d.file)
    g = group_summary(xml)
    groups.append(dict(stock_code=d.stock_code, group=g[0] if g else '', n_listed=g[1] if g else None, n_unlisted=g[2] if g else None))
    for kind, name, reg in affiliate_rows(xml):
        rows.append({'stock_code': d.stock_code, 'corp_name': CORP_NAME.get(d.stock_code, d.corp_name),
                     'group': g[0] if g else '', 'name': clean_name(name), 'name_raw': name, 'listed': kind == '상장',
                     'corp_reg_no': reg, 'rcept_no': d.rcept_no, 'report': d.report, 'base_date': d.base_date})
affiliates = pd.DataFrame(rows)
group_of = pd.DataFrame(groups)
print('사업보고서', len(docs), '/ 기업집단 이름이 있는 회사', (group_of.group != '').sum(),
      '/ 계열회사 줄', len(affiliates), '/ 상세표를 읽은 회사', affiliates.stock_code.nunique())

# 상장 계열회사가 어느 종목인지: 법인등록번호(대상 기업의 기업개황) → 이름(sector.json 종목명, DART 고유번호 목록의 법인명)
REG_TO_CODE = {}
for c in companies.stock_code:
    jur = read_saved(RAW / 'company' / f'{c}.json')['response'].get('jurir_no', '')
    if jur:
        REG_TO_CODE[jur] = c
# 한글로 풀어 쓴 영문 약칭("엘지전자㈜", "씨제이프레시웨이")을 종목명 표기로 바꿔 비교한다. 긴 것부터
LATIN = [('에스케이씨', 'skc'), ('에스케이', 'sk'), ('엘지', 'lg'), ('씨제이', 'cj'), ('지에스', 'gs'), ('케이티', 'kt'), ('엘에스', 'ls'),
         ('에이치디', 'hd'), ('디비', 'db'), ('케이지', 'kg'), ('에이치엘', 'hl'), ('엘엑스', 'lx'), ('에이치에스', 'hs')]


def aff_key(name):
    """계열회사 이름 비교 키: 법인 표기·공백·문장부호를 지우고, 앞의 한글 영문 약칭을 영문으로."""
    s = re.sub(r'\(주\)|㈜|주식회사|\(유\)|유한회사|[\s.,·()\-]', '', name or '').lower()
    for ko, en in LATIN:
        if s.startswith(ko):
            return en + s[len(ko):]
    return s


stocks = json.loads((ROOT.parent / 'market_data' / 'sector.json').read_text(encoding='utf-8'))['stocks']
code_by_key = {}
for c, v in stocks.items():
    code_by_key.setdefault(aff_key(v['security_name']), set()).add(c)
corp_zip = sorted((RAW / 'corpCode').glob('corpCode_*.zip'))[-1]
with zipfile.ZipFile(corp_zip) as z:
    for e in ET.fromstring(z.read(z.namelist()[0])).iter('list'):
        sc = (e.findtext('stock_code') or '').strip()
        if sc in stocks:
            code_by_key.setdefault(aff_key(e.findtext('corp_name')), set()).add(sc)


def affiliate_code(row):
    """상장 계열회사의 종목코드. 못 찾으면 ''."""
    if not row['listed']:
        return ''
    c = REG_TO_CODE.get(row['corp_reg_no'].replace('-', ''))
    if c:
        return c
    m = re.search(r'종목코드\s*([0-9A-Z]{6})', row['name_raw'])       # "씨제이프레시웨이(주) (종목코드 051500)"
    if m and m.group(1) in stocks:
        return m.group(1)
    cs = code_by_key.get(aff_key(row['name']), set())
    return next(iter(cs)) if len(cs) == 1 else ''


affiliates['affiliate_stock_code'] = affiliates.apply(affiliate_code, axis=1)
lst = affiliates[affiliates['listed']]
print('상장 계열회사 줄', len(lst), '/ 종목을 찾음', (lst.affiliate_stock_code != '').sum(),
      '/ 못 찾은 이름', lst[lst.affiliate_stock_code == ''].name.value_counts().head(15).to_dict())

affiliates.to_csv(OUT / 'affiliates.csv', index=False, encoding='utf-8-sig')
group_of.to_csv(OUT / 'affiliate_groups.csv', index=False, encoding='utf-8-sig')

사업보고서 1063 / 기업집단 이름이 있는 회사 883 / 계열회사 줄 53980 / 상세표를 읽은 회사 985


상장 계열회사 줄 3637 / 종목을 찾음 3396 / 못 찾은 이름 {'SK위탁관리부동산투자회사': 17, '삼성에프엔위탁관리부동산투자회사(주)': 16, '한화위탁관리부동산투자회사 주식회사': 11, '롯데위탁관리부동산투자회사㈜': 10, '(주)현대홈쇼핑': 9, 'HD현대인프라코어㈜': 8, '한솔인티큐브(주)': 6, 'SM ENTERTAINMENT JAPAN Co.,Ltd': 6, '에스지씨이앤씨(주)': 5, '(주)에스엠컬처앤콘텐츠': 5, '(주)에스엠라이프디자인그룹': 5, '(주)세아특수강': 4, '코오롱모빌리티그룹㈜': 3, 'MetaVia Inc': 3, '코오롱이앤피㈜': 3}


In [20]:
# ── 확인: 계열회사 표 예(SK하이닉스)와 기업집단 이름이 많이 나온 것 ──
display(affiliates[affiliates.stock_code == '000660'].head(8)[['group', 'name', 'listed', 'corp_reg_no', 'affiliate_stock_code']])
group_of[group_of.group != ''].group.value_counts().head(15)

,group,name,listed,corp_reg_no,affiliate_stock_code
750,에스케이,드림어스컴퍼니,True,110111-1637383,060570
751,에스케이,ISC,True,131111-0057876,095340
752,에스케이,SK(주),True,110111-0769583,034730
753,에스케이,SK가스,True,110111-0413247,018670
754,에스케이,SK네트웍스,True,130111-0005199,001740
755,에스케이,SK디스커버리,True,130111-0005727,006120
756,에스케이,SK디앤디,True,110111-3001685,210980
757,에스케이,SK바이오사이언스,True,131111-0523736,302440


group
에스케이     17
삼성       15
한화       11
현대자동차    10
롯데       10
효성        9
현대백화점     9
한진        8
한솔        7
씨제이       7
카카오       6
엘에스       6
두산        6
지에스       6
포스코       6
Name: count, dtype: int64